# identity_v2 — biographical identity under induction

`results/identity/identity_v2`, the uncapped re-run. **identity_v1 is dead data**: every
one of its 5,630 responses was collected at `max_tokens: 120` and 1,135 of them (20%,
six of seven models) stopped mid-answer at `finish_reason: length` and were judged as
though finished. Instruments now declare no cap, so v2 is a re-generation, not a re-score.

Two judges run at parse time, never during generation:

- **per-question hit** — WG's own rubrics, verbatim: did the answer name what the rule accepts?
- **stance** — the five-label grid in `data/sad/stance_grid.yaml`, scored blind (the judge is
  never told which persona was induced).

`identity_rate` is hits / scored, so a refusal or a failed call is absent from the
denominator rather than counted as a wrong answer.

In [ ]:
import json, math
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
RUN  = ROOT / 'results' / 'identity' / 'identity_v2'

def load(run: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(run.rglob('parsed.jsonl')):
        for line in p.read_text().splitlines():
            if not line.strip():
                continue
            r = json.loads(line)
            v = r.get('value') if isinstance(r.get('value'), dict) else {}
            rows.append({
                'model': r.get('model'), 'persona': r.get('persona'),
                'route': r.get('route'), 'item': r.get('item_id'),
                'sample': r.get('sample'), 'status': r.get('status'),
                'hit': v.get('hit'), 'stance': v.get('stance'),
                'answer': v.get('answer'), 'why': v.get('stance_why'),
            })
    return pd.DataFrame(rows)

df = load(RUN)
print(f'{len(df):,} rows  |  {df.model.nunique()} models  |  {df.groupby(["model","persona","route"]).ngroups} cells')
df.status.value_counts()

## Data quality first

Nothing below is worth reading until this is clean. Two failure modes have already cost a
full sweep each: truncation recorded as a finished answer, and an empty completion at
`finish_reason: stop` recorded as `ok`. Both now surface as `error`.

A cell that lost most of its rows still reports a rate — so check `n_scored`, not just the rate.

In [ ]:
q = (df.groupby(['model','persona','route'])
       .agg(n=('status','size'),
            errors=('status', lambda s: (s=='error').sum()),
            unparsed=('status', lambda s: (s=='unparsed').sum()))
       .reset_index())
q['loss'] = (q.errors + q.unparsed) / q.n
bad = q[q.loss > 0].sort_values('loss', ascending=False)
print(f'{len(bad)} of {len(q)} cells lost any rows\n')
bad.head(15)

## Identity rate by cell

Wilson interval, because a rate near 0 or 1 with n=50 has an asymmetric interval and the
normal approximation runs off the end of [0,1].

In [ ]:
def wilson(k, n, z=1.96):
    if not n: return (float('nan'), float('nan'))
    p = k/n; d = 1 + z*z/n; c = p + z*z/(2*n)
    m = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((c-m)/d, (c+m)/d)

scored = df[df.hit.notna()]
g = (scored.groupby(['model','persona','route'])
           .agg(n_scored=('hit','size'), hits=('hit','sum')).reset_index())
g['identity_rate'] = g.hits / g.n_scored
ci = [wilson(int(k), int(n)) for k, n in zip(g.hits, g.n_scored)]
g['ci_lo'] = [c[0] for c in ci]
g['ci_hi'] = [c[1] for c in ci]
g.sort_values(['model','persona','route']).head(30)

## Route comparison

The question §4.1 asks: does intervention depth order the routes? `system` puts the persona
in the system slot, `icl_k32` the same facts as conversation turns, `sft` the same facts in
the weights. `system_shuffled_k32` is the control — facts pooled across personas, so a cell
that scores above zero on it is scoring on something other than the target's biography.

In [ ]:
pivot = (g.pivot_table(index=['model','persona'], columns='route',
                       values='identity_rate')
          .round(3))
pivot

In [ ]:
(g.groupby('route')
  .agg(cells=('identity_rate','size'), mean=('identity_rate','mean'),
       min=('identity_rate','min'), max=('identity_rate','max'))
  .round(3).sort_values('mean', ascending=False))

## Stance — which entity answered

Independent of whether the answer was *right*. The uninduced `_base` cell is the floor:
it should be almost entirely `assistant`, and anything else there is the grid misfiring
rather than a persona effect.

In [ ]:
st = (df[df.stance.notna()]
      .pivot_table(index=['model','persona','route'], columns='stance',
                   values='item', aggfunc='size', fill_value=0))
st = st.div(st.sum(axis=1), axis=0).round(3)
st.loc[st.index.get_level_values('persona') == '_base']

In [ ]:
st.head(30)

## v1 vs v2 — what the cap cost

Only run if `identity_v1` is still on disk. This is the evidence for the re-run: the same
cells, the same questions, one collected at `max_tokens: 120` and one uncapped.

In [ ]:
V1 = ROOT / 'results' / 'identity' / 'identity_v1'
if V1.exists():
    import collections
    def finish(run):
        c = collections.Counter()
        for p in run.rglob('responses.jsonl'):
            for line in p.read_text().splitlines():
                if line.strip():
                    c[json.loads(line).get('finish_reason')] += 1
        return c
    for name, run in (('v1 (cap 120)', V1), ('v2 (uncapped)', RUN)):
        c = finish(run); tot = sum(c.values())
        cut = c.get('length', 0)
        print(f'{name:18s} {tot:6,d} responses   truncated {cut:5,d} ({cut/tot:.1%})')
else:
    print('identity_v1 not on disk')

## Reading an answer behind a number

Any rate in this notebook can be traced to the text that produced it. Change the filter.

In [ ]:
sel = df[(df.persona=='stalin') & (df.route=='sft') & (df.hit==False)]
for _, r in sel.head(4).iterrows():
    print(f"[{r.model}] {r.item}")
    print(f'   A: {str(r.answer)[:220]}')
    print(f'   stance={r.stance}  {str(r.why)[:110]}\n')